In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../'))

import numpy as np
import tensorflow as tf
import random, json

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)

In [ ]:
from src.pipeline.captioning_scratch import ImageCaptionerScratch
from src.evaluation.metrics import compute_sentence_bleu4
from src.evaluation.qualitative import visualize_captions, select_qualitative_samples

VOCAB_PATH         = '../../data/captions/vocab.json'
TEST_CAPTIONS_PATH = '../../data/captions/test_captions.json'
TEST_FEATURES_PATH = '../../data/features/test_features.npy'
TEST_IMAGES_DIR    = '../../data/images/test/'

BEST_RNN_WEIGHTS   = '../../weights/rnn_lstm/rnn_preinject_L?_H???.h5'   
BEST_LSTM_WEIGHTS  = '../../weights/rnn_lstm/lstm_preinject_L?_H???.h5' 

with open(TEST_CAPTIONS_PATH) as f:
    test_data = json.load(f)
test_image_names = list(test_data.keys())
test_references  = [test_data[name] for name in test_image_names]
test_image_paths = [os.path.join(TEST_IMAGES_DIR, name) for name in test_image_names]

test_features = np.load(TEST_FEATURES_PATH, allow_pickle=True).item()
if isinstance(test_features, dict):
    test_feat_matrix = np.stack([test_features[n] for n in test_image_names])
else:
    test_feat_matrix = test_features

In [ ]:
rnn_captioner = ImageCaptionerScratch('rnn')
rnn_captioner.load_weights('InceptionV3', BEST_RNN_WEIGHTS, VOCAB_PATH)
rnn_captions = rnn_captioner.predict_batch_from_features(test_feat_matrix)

lstm_captioner = ImageCaptionerScratch('lstm')
lstm_captioner.load_weights('InceptionV3', BEST_LSTM_WEIGHTS, VOCAB_PATH)
lstm_captions = lstm_captioner.predict_batch_from_features(test_feat_matrix)

print('Caption generation done.')

In [ ]:
scores_rnn  = [compute_sentence_bleu4(refs, hyp) for refs, hyp in zip(test_references, rnn_captions)]
scores_lstm = [compute_sentence_bleu4(refs, hyp) for refs, hyp in zip(test_references, lstm_captions)]

print(f'RNN  BLEU-4: mean={np.mean(scores_rnn):.4f}  std={np.std(scores_rnn):.4f}')
print(f'LSTM BLEU-4: mean={np.mean(scores_lstm):.4f}  std={np.std(scores_lstm):.4f}')

In [ ]:
selected = select_qualitative_samples(
    scores_rnn, scores_lstm,
    n_low=3, n_mid=4, n_high=3
)
print(f'Selected {len(selected)} images: {selected}')

In [ ]:
import os
os.makedirs('../../results/plots', exist_ok=True)

from src.evaluation.qualitative import visualize_captions

visualize_captions(
    image_paths=test_image_paths,
    rnn_captions=rnn_captions,
    lstm_captions=lstm_captions,
    references=test_references,
    scores_rnn=scores_rnn,
    scores_lstm=scores_lstm,
    selected_indices=selected,
    save_path='../../results/plots/qualitative_analysis.png',
)

## Poin Analisis Wajib

### 1. Pola RNN vs LSTM pada gambar dengan score rendah

### 2. Kemampuan memori jangka panjang

### 3. Kata yang berulang

### 4. Objek utama vs konteks/aksi